# Image Classification Using Convolutional Neural Networks (CNN)

**Programme:** Summer Training Programme on Data Science, Machine Learning & Agentic AI  
**Organised by:** Electronics & ICT Academy, IIT Roorkee

This notebook builds and evaluates a CNN for classifying images from the CIFAR-10 dataset into 10 categories.

## 1. Import libraries and set configuration

We use TensorFlow/Keras for the CNN, NumPy for array operations, Matplotlib for plots, and scikit-learn for the validation split and evaluation metrics.

In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
BATCH_SIZE = 64
EPOCHS = 25

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load the CIFAR-10 dataset

CIFAR-10 contains 60,000 RGB images of size 32×32 pixels. It has 10 balanced classes, with 50,000 training images and 10,000 test images.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train_full = y_train_full.squeeze()
y_test = y_test.squeeze()

print("Original training set:", x_train_full.shape, y_train_full.shape)
print("Test set:", x_test.shape, y_test.shape)

## 3. Normalize and create a validation set

Pixel values originally range from 0 to 255. Dividing by 255 scales them to the interval [0, 1]. We reserve 10% of the original training set for validation.

In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full,
)

print("Training:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## 4. Visualize sample images

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 7))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[y_train[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Data augmentation

Small random transformations expose the model to slightly different versions of the same image. This improves generalization and reduces overfitting.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name="data_augmentation",
)

## 6. Build the CNN

Convolutional layers learn local visual patterns such as edges, textures, and shapes. Pooling reduces spatial dimensions. Batch normalization stabilizes training, while dropout reduces overfitting.

In [ ]:
inputs = tf.keras.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)

x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)

x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)

x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)

x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.50)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="cifar10_cnn")
model.summary()

## 7. Compile the model

We use Adam optimization and sparse categorical cross-entropy because this is a 10-class classification problem with integer class labels.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 8. Train the model

Early stopping restores the model parameters from the best validation epoch. ReduceLROnPlateau lowers the learning rate if validation loss stops improving.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

## 9. Plot training history

In [ ]:
epochs_ran = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs_ran, history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_curve.png", dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history.history["loss"], label="Training Loss")
plt.plot(epochs_ran, history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=200)
plt.show()

## 10. Evaluate on the unseen test set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy (%): {test_accuracy * 100:.2f}%")

## 11. Classification report and confusion matrix

In [ ]:
probabilities = model.predict(x_test, verbose=0)
predictions = np.argmax(probabilities, axis=1)

print(classification_report(
    y_test,
    predictions,
    target_names=CLASS_NAMES,
    digits=4
))

cm = confusion_matrix(y_test, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("CIFAR-10 Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200)
plt.show()

## 12. Visualize sample predictions

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.choice(len(x_test), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, idx in zip(axes.ravel(), indices):
    ax.imshow(x_test[idx])
    true_name = CLASS_NAMES[y_test[idx]]
    pred_name = CLASS_NAMES[predictions[idx]]
    ax.set_title(f"True: {true_name}\nPred: {pred_name}")
    ax.axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_predictions.png", dpi=200)
plt.show()

## 13. Save the trained model

In [ ]:
model.save(MODELS_DIR / "cifar10_cnn.keras")
print("Model saved to:", MODELS_DIR / "cifar10_cnn.keras")

## 14. Predict one image

This helper function can be used to inspect individual predictions from the test set.

In [ ]:
def predict_image(image):
    image_batch = np.expand_dims(image, axis=0)
    probs = model.predict(image_batch, verbose=0)[0]
    pred_class = int(np.argmax(probs))
    confidence = float(np.max(probs))
    return CLASS_NAMES[pred_class], confidence

index = 0
predicted_class, confidence = predict_image(x_test[index])

plt.imshow(x_test[index])
plt.title(
    f"True: {CLASS_NAMES[y_test[index]]}\n"
    f"Predicted: {predicted_class} ({confidence:.2%})"
)
plt.axis("off")
plt.show()

## Conclusion

The project demonstrates an end-to-end image-classification pipeline using a CNN. The final test accuracy, per-class precision/recall/F1 scores, confusion matrix, and prediction examples are generated directly by the trained model and can be used in the project report.